# Pet Temperature EDA

이 노트북은 반려동물 관련 시설(동물병원, 애완용품점, 반려동물 놀이터 등) 데이터를 분석하여 '반려동물 온도(Pet Temperature)' 점수 산정 로직을 수립하기 위해 작성되었습니다.

## 분석 대상 데이터
1. **소상공인시장진흥공단 상가 정보**: 동물병원, 애완동물/애완용품 소매업
2. **서울시 반려동물 놀이터/카페 정보**: `animal_places.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 한글 폰트 설정 (Windows 기준)
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

# 데이터 로드 경로 설정
DATA_DIR = '../../../data/GraphDB_data'
STORE_CSV = os.path.join(DATA_DIR, 'store_data/소상공인시장진흥공단_상가(상권)정보_서울_cleaned.csv')
ANIMAL_CSV = os.path.join(DATA_DIR, 'animal/animal_places.csv')

## 1. 데이터 로드 및 전처리

In [ ]:
# 1. 상가 데이터 로드
# 필요한 컬럼만 로드하여 메모리 절약
cols = ['상호명', '상권업종대분류명', '상권업종중분류명', '상권업종소분류명', '위도', '경도']
df_store = pd.read_csv(STORE_CSV, usecols=cols)

print(f"Total Store Records: {len(df_store)}")
df_store.head()

In [ ]:
# 2. 반려동물 관련 업종 필터링
# '애완' 키워드가 포함된 업종 확인
df_pet_store = df_store[df_store['상권업종소분류명'].str.contains('애완|동물', na=False)]

# 카테고리별 개수 확인
print(df_pet_store['상권업종소분류명'].value_counts())

# 주요 카테고리: '동물병원', '애완동물/애완용품소매'
df_hospital = df_pet_store[df_pet_store['상권업종소분류명'] == '동물병원'].copy()
df_retail = df_pet_store[df_pet_store['상권업종소분류명'] == '애완동물/애완용품소매'].copy()

print(f"\nAnimal Hospitals: {len(df_hospital)}")
print(f"Pet Retail Stores: {len(df_retail)}")

In [ ]:
# 3. 반려동물 놀이터/카페 데이터 로드
df_play = pd.read_csv(ANIMAL_CSV)
print(f"Pet Playgrounds/Cafes: {len(df_play)}")
df_play.head()

## 2. 분포 시각화 (선택 사항)

In [ ]:
plt.figure(figsize=(10, 6))
plt.title('반려동물 관련 시설 분포')
plt.scatter(df_retail['경도'], df_retail['위도'], s=1, alpha=0.3, label='소매점', c='blue')
plt.scatter(df_hospital['경도'], df_hospital['위도'], s=3, alpha=0.5, label='동물병원', c='green')
plt.scatter(df_play['longitude'], df_play['latitude'], s=20, alpha=1.0, label='놀이터/카페', c='red', marker='*')
plt.legend()
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 3. 거리 기반 접근성 분석 및 점수 제안

### 분석 목표
- 각 매물(Here: `scripts/data_import/importers/neo4j_importers/property/property_importer.py` 로 임포트 된 Property 노드 가정)에 대해,
- 500m 반경 내에 시설이 얼마나 존재하는지 시뮬레이션 하거나,
- 시설의 밀도를 보고 적절한 가중치를 제안.

### 시설별 가중치 제안 (Draft)
다른 온도 점수(문화, 편의)와의 형평성을 고려하여 총점 100점 만점으로 설계.

| 시설 종류 | 희소성/중요도 | 제안 가중치 (개당) | 비고 |
| :--- | :--- | :--- | :--- |
| **동물병원** | 필수 시설, 비교적 흔함 | **20점** | 응급 상황 대비 중요 |
| **애완용품점** | 편의 시설, 흔함 | **5~10점** | 온라인 대체 가능성 |
| **반려동물 놀이터** | 매우 희소, 선호도 높음 | **30~40점** | 산책 및 여가 |
| **반려동물 동반 카페** | 희소 | **10~15점** | 여가 |

### 거리 감점 로직 (Distance Decay)
- `Score = Weight * (1 - distance / 500)`
- 500m 이내 선형 감점 적용.